In [1]:
from lcpy.calculators.bw_int import mpLCAer
import os
from lcpy.hvs.hvs import store_scenario_results
from lcpy.calculators.env_calc import fast_calculator
import pandas as pd
from lcpy.hvs.map_dicts import create_mapping, create_list_with_unique_activities
import numpy as np

The initial configuration remains the same with the simple LCA example

In [2]:
target_dir = "path_of_directory_where_to_store_results"
os.makedirs(target_dir, exist_ok=True)

In [3]:
brightway_configuration_dictionary = {
    "path_to_brightway_project": "path_to_folder_containing_the_bw_environment_and_packages_installed_there",
    "bw_project": "bw_project_name",
    "bw_database": "bw_project_database_name",
    "bw_biosphere": "bw_project_biosphere_database_name",
    "bw_ecoinvent": "ecoinvent_database_used_name"
}

In [4]:
methods_list = [
('TRACI v2.1', 'acidification', 'acidification potential (AP)'),
('TRACI v2.1', 'climate change', 'global warming potential (GWP100)'),
('TRACI v2.1', 'ecotoxicity: freshwater', 'ecotoxicity: freshwater'),
('TRACI v2.1', 'eutrophication', 'eutrophication potential'),
('TRACI v2.1', 'human toxicity: carcinogenic', 'human toxicity: carcinogenic'),
('TRACI v2.1', 'human toxicity: non-carcinogenic', 'human toxicity: non-carcinogenic'),
('TRACI v2.1', 'ozone depletion', 'ozone depletion potential (ODP)'),
('TRACI v2.1', 'particulate matter formation', 'particulate matter formation potential (PMFP)'),
('TRACI v2.1', 'photochemical oxidant formation', 'maximum incremental reactivity (MIR)'),
]

method_units_list = ['kg SO2-Eq',
 'kg CO2-Eq',
 'CTUe',
 'kg N-Eq',
 'CTUh',
 'CTUh',
 'kg CFC-11-Eq',
 'kg PM2.5-Eq',
 'kg O3-Eq',
]

In [5]:
methods_gp = methods_list[:]
impact_categories_names = ['AP', 'GWP100', 'ECFW', 'EP', 'HTC', 'HTNC', 'ODP', 'PMFP', 'MIR']

In contrast to the simple LCA we now set the number of scenarios to a high number that basically represents the MC loops

In [6]:
scenarios = 30000
timeframe = 1 #operational lifetime after construction
time_step = 1
construction_years = 0
number_of_infrastructure_processes = 0

The parametric model should now be set to include uncertainty for the MC loops. This is done below, by setting a number of parameters to raneg with some probability distributions

In [7]:
keys_nuclear_power_generation = {
    'PWR': 'bw_key_pointing_to_relevant_process',
    'BWR': 'bw_key_pointing_to_relevant_process'
}

keys_fossil_power_generation = {
    'NG_ccpp': 'bw_key_pointing_to_relevant_process',
    'NG_convpp': 'bw_key_pointing_to_relevant_process',
    'NG_cogen_conv': 'bw_key_pointing_to_relevant_process',
    'NG_cogen_cc': 'bw_key_pointing_to_relevant_process',
}

keys_res_power_generation = {
    'Hydro': 'bw_key_pointing_to_relevant_process',
    'DGE': 'bw_key_pointing_to_relevant_process',
    'Wind': 'bw_key_pointing_to_relevant_process',
}

keys_total_power_generation = {
    'Nuclear': '',
    'Fossil': '',
    'RES': '',
}

In [8]:
key_list_sub_processes = [keys_nuclear_power_generation, keys_fossil_power_generation, keys_res_power_generation]
mapping_names = create_mapping(keys_total_power_generation, key_list_sub_processes)
unique_activities = create_list_with_unique_activities(key_list_sub_processes)
my_lca = mpLCAer(4, methods_gp, brightway_configuration_dictionary)
my_lca.import_isolated_environment()
my_lca.lca_calculations(mapping_names)

In [10]:
def model(BWR_capacity, PWR_capacity, NG_ccpp_capacity, NG_cogen_conv_capacity, NG_convpp_capacity, NG_cogen_cc_capacity,
     Hydro_pumped_capacity, DGE_capacity, Wind_1_3_capacity):

    nuclear_capacity = BWR_capacity + PWR_capacity
    ng_capacity = NG_ccpp_capacity + NG_convpp_capacity + NG_cogen_conv_capacity + NG_cogen_cc_capacity
    res_capacity = Hydro_pumped_capacity + DGE_capacity + Wind_1_3_capacity
    total_capacity = nuclear_capacity + ng_capacity + res_capacity

    nuclear_exchanges_amounts = [
        PWR_capacity / nuclear_capacity,
        BWR_capacity / nuclear_capacity
    ]


    fossil_exchanges_amounts = [
        NG_ccpp_capacity / ng_capacity,
        NG_convpp_capacity / ng_capacity,
        NG_cogen_conv_capacity / ng_capacity,
        NG_cogen_cc_capacity / ng_capacity,
    ]

    res_exchanges_amounts = [
        Hydro_pumped_capacity / res_capacity,
        DGE_capacity / res_capacity,
        Wind_1_3_capacity / res_capacity,
    ]

    mp_exchanges_amounts = [
        nuclear_capacity / total_capacity,
        ng_capacity / total_capacity,
        res_capacity / total_capacity,
    ]
    exchanges_list_sp = [nuclear_exchanges_amounts, fossil_exchanges_amounts, res_exchanges_amounts]
    mapping_exchanges = create_mapping(keys_total_power_generation, exchanges_list_sp)
    my_calculator = fast_calculator()

    my_calculator.calculation_static_lcia(mapping_exchanges, my_lca.unit_impacts)
    my_calculator.calculation_impact_senarios(mp_exchanges_amounts, my_calculator.impact, 'Electricity production')

    gsa_result = my_calculator.total_impact['Electricity production'][:,:,0].T

    return gsa_result

In [11]:
problem = {
    'num_vars': 9,
    'names': ['BWR_capacity', 'PWR_capacity', 'NG_ccpp_capacity',
              'NG_cogen_conv_capacity', 'NG_convpp_capacity', 'NG_cogen_cc_capacity',
              'Hydro_pumped_capacity', 'DGE_capacity', 'Wind_1_3_capacity'],
    'bounds': [[1200, 1800],
               [700, 900],
               [350, 450],
               [600, 800],
               [560, 700],
               [230, 280],
               [320, 579],
               [55, 115],
               [900, 1100],
               ]
}
from SALib.sample import sobol
param_values = sobol.sample(problem, 2**10, calc_second_order=True)

In [12]:
BWR_capacity, PWR_capacity, NG_ccpp_capacity, NG_cogen_conv_capacity, NG_convpp_capacity, NG_cogen_cc_capacity, Hydro_pumped_capacity, DGE_capacity, Wind_1_3_capacity = [param_values[:,i].reshape(param_values[:,i].shape[0], 1) for i in range(param_values.shape[1])]

In [13]:
Y = model(BWR_capacity, PWR_capacity, NG_ccpp_capacity, NG_cogen_conv_capacity, NG_convpp_capacity, NG_cogen_cc_capacity, Hydro_pumped_capacity, DGE_capacity, Wind_1_3_capacity)

In [14]:
from SALib.analyze import sobol
Si = sobol.analyze(problem, Y[:,0])

In [15]:
parameter_names = problem['names']

In [16]:
df_S1 = pd.DataFrame(index=parameter_names,
                     columns=impact_categories_names,
                     dtype=float)
df_S2 = pd.DataFrame(index=parameter_names, columns=impact_categories_names)
df_ST = pd.DataFrame(index=parameter_names, columns=impact_categories_names)

In [17]:
for j, cat in enumerate(impact_categories_names):
    Si = sobol.analyze(problem, Y[:, j], calc_second_order=True)
    df_S1[cat] = Si['S1']
    df_S2[cat] = Si['S2']
    df_ST[cat] = Si['ST']

In [19]:
df_S1

,AP,GWP100,ECFW,EP,HTC,HTNC,ODP,PMFP,MIR
BWR_capacity,0.298322,0.509642,0.319320,0.027639,0.424870,0.092553,0.531901,0.076913,0.400010
PWR_capacity,0.033819,0.056806,0.038863,0.004095,0.049072,0.006560,0.060573,0.010262,0.045216
NG_ccpp_capacity,0.000309,0.011183,0.011599,0.003538,0.001337,0.010780,0.017358,0.001854,0.001546
NG_cogen_conv_capacity,0.034487,0.121374,0.001225,0.000731,0.000498,0.020728,0.172474,0.001271,0.071395
NG_convpp_capacity,0.031017,0.084006,0.018050,0.001151,0.000038,0.019538,0.084228,-0.000025,0.074462
NG_cogen_cc_capacity,0.000215,0.003873,0.002667,0.000749,0.000179,0.002687,0.005826,0.000273,0.000695
Hydro_pumped_capacity,0.574785,0.157934,0.334519,0.920246,0.427451,0.819052,0.063790,0.805889,0.363312
DGE_capacity,0.000233,0.002406,-0.000092,0.027082,0.024679,0.000271,0.004842,0.078646,0.000644
Wind_1_3_capacity,0.027502,0.053949,0.271997,0.013403,0.070706,0.026334,0.059975,0.023587,0.043757


# Contributions:

- Show how to do it for the modular approach
- make it for emissions of an environmental flow
- change sampling strategies to allow for more probability distributions